# Bachelorarbeit – Automatische Klassifikation von Brustkrebs in Mammographie-Daten
**Edanur Gür | MCI Innsbruck | RSNA Screening Mammography Dataset**

## Struktur
1. Imports & Konfiguration
2. Hilfsfunktionen (Preprocessing)
3. Daten laden, Datei-Check & Balancierung
4. Dataset & DataLoader
5. Modell A – Einfaches CNN von Null
6. Modell B – Transfer Learning (ResNet18)
7. Training & Validierung
8. Evaluation (ROC, Confusion Matrix, AUC)
9. Metadaten-Analyse A – Performance nach Altersgruppe
10. Metadaten-Analyse B – Alter als zusätzlicher Modell-Input
11. PCA Visualisierung
12. Architektur-Vergleich

## 1. Imports & Konfiguration

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import pydicom
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

from sklearn.metrics import (
    roc_curve, auc, confusion_matrix, classification_report
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ── Konfiguration ─────────────────────────────────────────────────────────────
BILD_GROESSE  = 256
BATCH_SIZE    = 16
EPOCHEN       = 15
LERNRATE      = 1e-3
VAL_ANTEIL    = 0.2        # 20% Validierung, 80% Training
ZUFALLSZAHL   = 42         # Für Reproduzierbarkeit
BILDER_ORDNER = 'train_images'
CSV_PFAD      = 'train.csv'
MODELL_PFAD_A = 'modell_cnn_scratch.pth'
MODELL_PFAD_B = 'modell_resnet18.pth'
MODELL_PFAD_C = 'modell_resnet18_multimodal.pth'

torch.manual_seed(ZUFALLSZAHL)
np.random.seed(ZUFALLSZAHL)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Gerät: {DEVICE.type.upper()}')

## 2. Hilfsfunktionen (Preprocessing)

In [ ]:
def lade_und_verarbeite_dicom(bild_pfad: str, zielgroesse: int = BILD_GROESSE) -> np.ndarray:
    """
    Lädt eine DICOM-Datei und gibt ein vorverarbeitetes uint8-Array zurück.
    Schritte: Invertierung (MONOCHROME1) → Normalisierung → Cropping → Resize
    """
    dicom = pydicom.dcmread(bild_pfad)
    bild  = dicom.pixel_array.astype(np.float32)

    # Invertieren wenn Hintergrund weiß (MONOCHROME1)
    if getattr(dicom, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        bild = bild.max() - bild

    # Normalisierung auf [0, 255]
    bild_min, bild_max = bild.min(), bild.max()
    if bild_max > bild_min:
        bild = (bild - bild_min) / (bild_max - bild_min)
    bild = (bild * 255).astype(np.uint8)

    # Schwarze Ränder wegschneiden
    maske = bild > 15
    if maske.any():
        zeilen  = np.where(maske.any(axis=1))[0]
        spalten = np.where(maske.any(axis=0))[0]
        bild = bild[zeilen[0]:zeilen[-1]+1, spalten[0]:spalten[-1]+1]

    bild = cv2.resize(bild, (zielgroesse, zielgroesse))
    return bild


def zeige_vorher_nachher(bild_pfad: str):
    """Zeigt Original und vorverarbeitetes Bild nebeneinander."""
    original    = pydicom.dcmread(bild_pfad).pixel_array
    verarbeitet = lade_und_verarbeite_dicom(bild_pfad)
    fig, achsen = plt.subplots(1, 2, figsize=(10, 5))
    achsen[0].imshow(original,    cmap='gray'); achsen[0].set_title('Original');            achsen[0].axis('off')
    achsen[1].imshow(verarbeitet, cmap='gray'); achsen[1].set_title('Vorverarbeitet (KI-Ready)'); achsen[1].axis('off')
    plt.tight_layout(); plt.show()


# Kurzer Test
test_pfad = 'train_images/10006/462822612.dcm'
if os.path.exists(test_pfad):
    zeige_vorher_nachher(test_pfad)
else:
    print(f'Testbild nicht gefunden: {test_pfad}')

## 3. Daten laden, Datei-Check & Balancierung

In [ ]:
# ── CSV laden ─────────────────────────────────────────────────────────────────
df_roh = pd.read_csv(CSV_PFAD)
print(f'Einträge in CSV gesamt: {len(df_roh):,}')
print(f'Gesund (0): {(df_roh["cancer"]==0).sum():,}  |  Krebs (1): {(df_roh["cancer"]==1).sum():,}')
display(df_roh.head())

# ── Datei-Check: nur Bilder die wirklich im Ordner vorhanden sind ──────────────
# Neue Patienten die du in train_images legst UND in der CSV stehen
# werden hier automatisch gefunden – du musst nichts extra tun.
print('\nPrüfe welche Bilddateien vorhanden sind...')

def bild_pfad_existiert(zeile):
    pfad = f"{BILDER_ORDNER}/{zeile['patient_id']}/{zeile['image_id']}.dcm"
    return os.path.exists(pfad)

df_roh['datei_vorhanden'] = df_roh.apply(bild_pfad_existiert, axis=1)
df_verfuegbar = df_roh[df_roh['datei_vorhanden']].copy().reset_index(drop=True)

print(f'Bilder gefunden:     {len(df_verfuegbar):,}')
print(f'Bilder nicht gefunden: {(~df_roh["datei_vorhanden"]).sum():,}')
print(f'\nVerfügbar – Gesund: {(df_verfuegbar["cancer"]==0).sum():,}  |  Krebs: {(df_verfuegbar["cancer"]==1).sum():,}')

In [ ]:
# ── Balancierung ──────────────────────────────────────────────────────────────
# Der RSNA Datensatz ist stark unbalanciert (~98% Gesund, ~2% Krebs).
# Wir balancieren auf 50/50 damit das Modell beide Klassen gleich gut lernt.

krebs_df  = df_verfuegbar[df_verfuegbar['cancer'] == 1]
gesund_df = df_verfuegbar[df_verfuegbar['cancer'] == 0]
n_krebs   = len(krebs_df)

print(f'Krebs-Fälle:  {n_krebs:,}')
print(f'Gesund-Fälle: {len(gesund_df):,}')

# 50/50 balanciert
gesund_sampled = gesund_df.sample(n=n_krebs, random_state=ZUFALLSZAHL)
df = pd.concat([krebs_df, gesund_sampled]).sample(
    frac=1, random_state=ZUFALLSZAHL).reset_index(drop=True)

print(f'\nBalancierter Datensatz: {len(df):,} Bilder (50% Krebs / 50% Gesund)')

# Visualisierung Vorher / Nachher
fig, achsen = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Klassenverteilung', fontsize=13, fontweight='bold')

for ax, daten, titel in zip(
    achsen,
    [df_verfuegbar, df],
    ['Original (unbalanciert)', 'Balanciert (50/50)']
):
    werte  = daten['cancer'].value_counts().sort_index()
    farben = ['steelblue', 'tomato']
    ax.bar(['Gesund (0)', 'Krebs (1)'], werte.values, color=farben, edgecolor='white')
    ax.set_title(titel)
    ax.set_ylabel('Anzahl Bilder')
    for i, v in enumerate(werte.values):
        ax.text(i, v + 5, f'{v:,}', ha='center', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Altersgruppen für spätere Metadaten-Analyse vorbereiten
df['altersgruppe'] = pd.cut(
    df['age'],
    bins=[0, 40, 50, 60, 70, 120],
    labels=['<40', '40-50', '50-60', '60-70', '>70']
)

## 4. Dataset & DataLoader

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
# ImageNet Normalisierung für ResNet18 (3 Kanäle)
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transforms_1kanal = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
val_transforms_1kanal = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
train_transforms_3kanal = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])
val_transforms_3kanal = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])


class MammografieDataset(Dataset):
    """
    Lädt DICOM-Mammographie-Bilder + Metadaten.
    fuer_resnet18=True  → 3 Kanäle (ResNet18 braucht RGB)
    fuer_resnet18=False → 1 Kanal  (eigenes CNN)
    mit_alter=True      → gibt Alter normalisiert zurück (für multimodales Modell)
    """
    def __init__(self, dataframe, basis_ordner,
                 transform=None, fuer_resnet18=False, mit_alter=False):
        self.df           = dataframe.reset_index(drop=True)
        self.basis_ordner = basis_ordner
        self.transform    = transform
        self.fuer_resnet18 = fuer_resnet18
        self.mit_alter    = mit_alter
        # Alter normalisieren (Mittelwert ~60, Std ~10) für stabiles Training
        self.alter_mean   = self.df['age'].mean()
        self.alter_std    = self.df['age'].std() + 1e-8

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        zeile     = self.df.iloc[idx]
        bild_pfad = f"{self.basis_ordner}/{zeile['patient_id']}/{zeile['image_id']}.dcm"
        label     = torch.tensor(zeile['cancer'], dtype=torch.float32)

        bild = lade_und_verarbeite_dicom(bild_pfad)

        if self.fuer_resnet18:
            # ResNet18 erwartet 3 Kanäle → Graustufen 3x wiederholen
            bild = np.stack([bild, bild, bild], axis=-1)  # (H, W, 3)
            bild_tensor = self.transform(bild) if self.transform else \
                torch.tensor(bild.transpose(2,0,1), dtype=torch.float32) / 255.0
        else:
            bild_tensor = self.transform(bild) if self.transform else \
                torch.tensor(bild[np.newaxis], dtype=torch.float32) / 255.0

        # Metadaten
        meta = {
            'patient_id':  zeile['patient_id'],
            'age':         zeile.get('age', -1),
            'laterality':  zeile.get('laterality', ''),
            'view':        zeile.get('view', ''),
            'altersgruppe': str(zeile.get('altersgruppe', '')),
        }

        if self.mit_alter:
            alter_norm = torch.tensor(
                [(zeile.get('age', self.alter_mean) - self.alter_mean) / self.alter_std],
                dtype=torch.float32
            )
            return bild_tensor, label, alter_norm, meta

        return bild_tensor, label, meta


def erstelle_dataloaders(dataframe, fuer_resnet18=False, mit_alter=False,
                          batch_size=BATCH_SIZE):
    """
    Teilt Daten in Train (80%) / Val (20%) auf.
    Gibt (loader_train, loader_val, df_train, df_val) zurück.
    df_train/df_val werden für die Metadaten-Analyse gebraucht.
    """
    n_val   = int(len(dataframe) * VAL_ANTEIL)
    n_train = len(dataframe) - n_val
    idx_train, idx_val = random_split(
        range(len(dataframe)), [n_train, n_val],
        generator=torch.Generator().manual_seed(ZUFALLSZAHL)
    )
    df_train = dataframe.iloc[list(idx_train)]
    df_val   = dataframe.iloc[list(idx_val)]

    t_train = train_transforms_3kanal if fuer_resnet18 else train_transforms_1kanal
    t_val   = val_transforms_3kanal   if fuer_resnet18 else val_transforms_1kanal

    ds_train = MammografieDataset(df_train, BILDER_ORDNER, t_train, fuer_resnet18, mit_alter)
    ds_val   = MammografieDataset(df_val,   BILDER_ORDNER, t_val,   fuer_resnet18, mit_alter)

    loader_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True,
                               num_workers=2, pin_memory=True)
    loader_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False,
                               num_workers=2, pin_memory=True)

    print(f'Train: {len(ds_train):,} | Val: {len(ds_val):,} Bilder')
    return loader_train, loader_val, df_train, df_val


# DataLoader für Modell A (1 Kanal) und Modell B (3 Kanäle)
loader_train_a, loader_val_a, df_train, df_val = erstelle_dataloaders(
    df, fuer_resnet18=False)
loader_train_b, loader_val_b, _, _ = erstelle_dataloaders(
    df, fuer_resnet18=True)

## 5. Modell A – Einfaches CNN von Null

In [ ]:
class EinfachesBrustkrebsCNN(nn.Module):
    """
    Kleines CNN als Baseline (Modell A).
    3 Conv-Schichten + AdaptiveAvgPool → funktioniert mit jeder Eingabegröße.
    Dropout gegen Overfitting.
    """
    def __init__(self, in_kanale=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_kanale, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),        nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),        nn.ReLU(), nn.MaxPool2d(2),
        )
        self.avgpool    = nn.AdaptiveAvgPool2d((8, 8))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1), nn.Sigmoid()
        )

    def forward(self, x):
        return self.classifier(self.avgpool(self.features(x)))


modell_a = EinfachesBrustkrebsCNN(in_kanale=1).to(DEVICE)
params_a  = sum(p.numel() for p in modell_a.parameters() if p.requires_grad)
print(f'Modell A – CNN von Null: {params_a:,} trainierbare Parameter')
print(modell_a)

## 6. Modell B – Transfer Learning (ResNet18)

In [ ]:
def erstelle_resnet18(einfrieren=True):
    """
    ResNet18 mit vortrainierten ImageNet-Gewichten.
    Architektur: Input → Conv1 → MaxPool → 4x Residual Blocks → AvgPool → FC
    Skip Connections in jedem Block verhindern das Vanishing Gradient Problem.

    einfrieren=True  → nur der neue FC-Kopf wird trainiert (Phase 1)
    einfrieren=False → alle Schichten fine-tunen (Phase 2, optional)
    """
    modell = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if einfrieren:
        for param in modell.parameters():
            param.requires_grad = False

    # FC Layer ersetzen: Linear(512, 1000) → Dropout + Linear(512, 1) + Sigmoid
    modell.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(modell.fc.in_features, 1),  # in_features = 512
        nn.Sigmoid()
    )
    return modell.to(DEVICE)


modell_b    = erstelle_resnet18(einfrieren=True)
trainierbar = sum(p.numel() for p in modell_b.parameters() if p.requires_grad)
gesamt      = sum(p.numel() for p in modell_b.parameters())
print(f'Modell B – ResNet18 (Transfer Learning)')
print(f'Trainierbare Parameter: {trainierbar:,} / {gesamt:,} gesamt')
print(f'Eingefroren:            {gesamt - trainierbar:,} Parameter')

## 7. Training & Validierung

In [ ]:
def trainiere_eine_epoche(modell, loader, optimierer, kriterium, mit_alter=False):
    modell.train()
    gesamt_loss = 0.0
    for batch in loader:
        if mit_alter:
            bilder, labels, alter, _ = batch
            alter = alter.to(DEVICE)
        else:
            bilder, labels, _ = batch
            alter = None

        bilder = bilder.to(DEVICE)
        labels = labels.unsqueeze(1).to(DEVICE)

        optimierer.zero_grad()
        vorhersagen = modell(bilder, alter) if mit_alter else modell(bilder)
        loss        = kriterium(vorhersagen, labels)
        loss.backward()
        optimierer.step()
        gesamt_loss += loss.item()

    return gesamt_loss / len(loader)


def validiere(modell, loader, kriterium, mit_alter=False):
    modell.eval()
    gesamt_loss = 0.0
    alle_labels, alle_preds = [], []

    with torch.no_grad():
        for batch in loader:
            if mit_alter:
                bilder, labels, alter, _ = batch
                alter = alter.to(DEVICE)
            else:
                bilder, labels, _ = batch
                alter = None

            bilder = bilder.to(DEVICE)
            labels = labels.unsqueeze(1).to(DEVICE)

            vorhersagen  = modell(bilder, alter) if mit_alter else modell(bilder)
            loss         = kriterium(vorhersagen, labels)
            gesamt_loss += loss.item()
            alle_labels.extend(labels.cpu().numpy())
            alle_preds.extend(vorhersagen.cpu().numpy())

    return gesamt_loss / len(loader), np.array(alle_labels), np.array(alle_preds)


def trainiere_modell(modell, loader_train, loader_val, epochen=EPOCHEN,
                     lernrate=LERNRATE, speicher_pfad=None, mit_alter=False):
    """
    Komplette Trainingsschleife.
    - BCELoss als Verlustfunktion
    - Adam Optimizer
    - ReduceLROnPlateau: halbiert LR wenn Val Loss 5 Epochen nicht besser wird
    - Bestes Modell wird automatisch gespeichert
    """
    kriterium  = nn.BCELoss()
    optimierer = optim.Adam(
        filter(lambda p: p.requires_grad, modell.parameters()), lr=lernrate)
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
        optimierer, mode='min', factor=0.5, patience=5, verbose=True)

    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    bester_val_loss = float('inf')

    for epoche in range(epochen):
        train_loss               = trainiere_eine_epoche(
            modell, loader_train, optimierer, kriterium, mit_alter)
        val_loss, labels, preds  = validiere(modell, loader_val, kriterium, mit_alter)

        fpr, tpr, _ = roc_curve(labels, preds)
        val_auc     = auc(fpr, tpr)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)
        scheduler.step(val_loss)

        if speicher_pfad and val_loss < bester_val_loss:
            bester_val_loss = val_loss
            torch.save(modell.state_dict(), speicher_pfad)
            print(f'  → Gespeichert (Val Loss: {val_loss:.4f})')

        print(f'Epoche {epoche+1:>2}/{epochen} | '
              f'Train: {train_loss:.4f} | Val: {val_loss:.4f} | AUC: {val_auc:.4f}')

    return history

In [ ]:
# ── Modell A trainieren ───────────────────────────────────────────────────────
print('=' * 60)
print('TRAINING MODELL A – CNN von Null')
print('=' * 60)
history_a = trainiere_modell(
    modell_a, loader_train_a, loader_val_a, speicher_pfad=MODELL_PFAD_A)

# ── Modell B trainieren ───────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('TRAINING MODELL B – ResNet18 (Transfer Learning)')
print('=' * 60)
history_b = trainiere_modell(
    modell_b, loader_train_b, loader_val_b, speicher_pfad=MODELL_PFAD_B)

## 8. Evaluation (ROC, Confusion Matrix, AUC)

In [ ]:
def lade_bestes_modell(modell, pfad):
    if os.path.exists(pfad):
        modell.load_state_dict(torch.load(pfad, map_location=DEVICE))
        print(f'Gewichte geladen: {pfad}')
    return modell


def evaluiere_modell(modell, loader, modell_name='Modell',
                     schwellwert=0.5, mit_alter=False):
    """
    Vollständige Evaluation:
    ROC Kurve + AUC | Confusion Matrix | Precision, Recall, F1
    Gibt (auc_wert, fpr, tpr, cm, labels_arr, preds_arr) zurück
    → labels und preds werden für Metadaten-Analyse weiterverwendet.
    """
    modell.eval()
    alle_labels, alle_preds, alle_meta = [], [], []

    with torch.no_grad():
        for batch in loader:
            if mit_alter:
                bilder, labels, alter, meta = batch
                vorhersagen = modell(bilder.to(DEVICE), alter.to(DEVICE))
            else:
                bilder, labels, meta = batch
                vorhersagen = modell(bilder.to(DEVICE))

            alle_labels.extend(labels.numpy())
            alle_preds.extend(vorhersagen.cpu().numpy().flatten())
            # Metadaten sammeln für spätere Altersgruppen-Analyse
            for i in range(len(labels)):
                alle_meta.append({
                    k: (v[i].item() if hasattr(v[i], 'item') else v[i])
                    for k, v in meta.items()
                })

    labels_arr = np.array(alle_labels)
    preds_arr  = np.array(alle_preds)
    klassen    = (preds_arr >= schwellwert).astype(int)

    fpr, tpr, _ = roc_curve(labels_arr, preds_arr)
    roc_auc     = auc(fpr, tpr)

    fig, achsen = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Evaluation: {modell_name}', fontsize=14, fontweight='bold')

    # ROC Kurve
    achsen[0].plot(fpr, tpr, color='darkorange', lw=2,
                   label=f'AUC = {roc_auc:.3f}')
    achsen[0].plot([0,1],[0,1],'k--', lw=1, label='Zufall')
    achsen[0].set_xlabel('False Positive Rate')
    achsen[0].set_ylabel('True Positive Rate (Sensitivität)')
    achsen[0].set_title('ROC Kurve')
    achsen[0].legend(); achsen[0].grid(alpha=0.3)

    # Confusion Matrix
    cm = confusion_matrix(labels_arr, klassen)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=achsen[1],
                xticklabels=['Gesund (0)', 'Krebs (1)'],
                yticklabels=['Gesund (0)', 'Krebs (1)'])
    achsen[1].set_xlabel('Vorhergesagt')
    achsen[1].set_ylabel('Tatsächlich')
    achsen[1].set_title('Confusion Matrix')

    plt.tight_layout(); plt.show()

    print(f'AUC: {roc_auc:.4f}')
    print(classification_report(labels_arr, klassen,
                                  target_names=['Gesund', 'Krebs']))

    return roc_auc, fpr, tpr, cm, labels_arr, preds_arr, alle_meta


# Beste Modelle laden und evaluieren
modell_a = lade_bestes_modell(modell_a, MODELL_PFAD_A)
modell_b = lade_bestes_modell(modell_b, MODELL_PFAD_B)

auc_a, fpr_a, tpr_a, cm_a, labels_a, preds_a, meta_a = evaluiere_modell(
    modell_a, loader_val_a, 'Modell A – CNN von Null')
auc_b, fpr_b, tpr_b, cm_b, labels_b, preds_b, meta_b = evaluiere_modell(
    modell_b, loader_val_b, 'Modell B – ResNet18')

## 9. Metadaten-Analyse A – Performance nach Altersgruppe

In [ ]:
# ── 9a. Altersverteilung im Datensatz ─────────────────────────────────────────
fig, achsen = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Altersverteilung im Datensatz', fontsize=13, fontweight='bold')

for label, farbe, name in [(0, 'steelblue', 'Gesund'), (1, 'tomato', 'Krebs')]:
    alter = df[df['cancer'] == label]['age'].dropna()
    achsen[0].hist(alter, bins=20, alpha=0.6, color=farbe, label=name)
achsen[0].set_xlabel('Alter'); achsen[0].set_ylabel('Anzahl')
achsen[0].set_title('Altersverteilung')
achsen[0].legend(); achsen[0].grid(alpha=0.3)

gesund_alter = df[df['cancer']==0]['age'].dropna()
krebs_alter  = df[df['cancer']==1]['age'].dropna()
achsen[1].boxplot([gesund_alter, krebs_alter], labels=['Gesund', 'Krebs'],
                   patch_artist=True,
                   boxprops=dict(facecolor='lightblue', color='navy'))
achsen[1].set_ylabel('Alter')
achsen[1].set_title('Altersverteilung (Boxplot)')
achsen[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Ø Alter Gesund: {gesund_alter.mean():.1f} | Ø Alter Krebs: {krebs_alter.mean():.1f}')
print('\nKrebs-Prävalenz nach Altersgruppe:')
print(df.groupby('altersgruppe')['cancer'].mean().round(4))

In [ ]:
# ── 9b. Modell-Performance nach Altersgruppe ──────────────────────────────────
# Das ist genau was dein Betreuer meinte:
# Gibt es Altersgruppen bei denen das Modell besser/schlechter ist?

def performance_nach_altersgruppe(labels, preds, meta, modell_name):
    """
    Berechnet AUC pro Altersgruppe und zeigt ob das Modell
    bei bestimmten Altersgruppen systematisch besser/schlechter ist.
    """
    df_eval = pd.DataFrame({
        'label':        labels,
        'pred':         preds,
        'altersgruppe': [m['altersgruppe'] for m in meta],
        'age':          [m['age'] for m in meta]
    })

    gruppen    = ['<40', '40-50', '50-60', '60-70', '>70']
    auc_werte  = []
    n_werte    = []

    print(f'\n{modell_name} – AUC nach Altersgruppe:')
    print(f'{"Gruppe":<12} {"N":>6} {"AUC":>8}')
    print('-' * 30)

    for gruppe in gruppen:
        teilmenge = df_eval[df_eval['altersgruppe'] == gruppe]
        n = len(teilmenge)
        if n < 10 or teilmenge['label'].nunique() < 2:
            auc_werte.append(None)
            n_werte.append(n)
            print(f'{gruppe:<12} {n:>6}  zu wenig Daten')
            continue
        fpr_g, tpr_g, _ = roc_curve(teilmenge['label'], teilmenge['pred'])
        auc_g = auc(fpr_g, tpr_g)
        auc_werte.append(auc_g)
        n_werte.append(n)
        print(f'{gruppe:<12} {n:>6} {auc_g:>8.4f}')

    # Balkendiagramm
    gueltige = [(g, a, n) for g, a, n in zip(gruppen, auc_werte, n_werte) if a is not None]
    if gueltige:
        g_namen, g_auc, g_n = zip(*gueltige)
        fig, ax = plt.subplots(figsize=(9, 4))
        balken = ax.bar(g_namen, g_auc, color='steelblue', edgecolor='white')
        ax.axhline(y=0.5, color='red', linestyle='--', lw=1, label='Zufall (AUC=0.5)')
        ax.set_xlabel('Altersgruppe'); ax.set_ylabel('AUC')
        ax.set_title(f'{modell_name} – AUC nach Altersgruppe')
        ax.set_ylim(0, 1); ax.legend(); ax.grid(axis='y', alpha=0.3)
        for balken_i, n in zip(balken, g_n):
            ax.text(balken_i.get_x() + balken_i.get_width()/2,
                    balken_i.get_height() + 0.01,
                    f'n={n}', ha='center', fontsize=8)
        plt.tight_layout(); plt.show()

    return df_eval


df_eval_a = performance_nach_altersgruppe(labels_a, preds_a, meta_a, 'Modell A – CNN von Null')
df_eval_b = performance_nach_altersgruppe(labels_b, preds_b, meta_b, 'Modell B – ResNet18')

## 10. Metadaten-Analyse B – Alter als zusätzlicher Modell-Input (Multimodal)

In [ ]:
# ── Multimodales Modell ───────────────────────────────────────────────────────
# Idee: Bild durch ResNet18 → 512 Features
#        + normalisiertes Alter (1 Zahl)
#        → zusammen in finalen Classifier
# Das nennt sich 'multimodales Modell' – kombiniert Bild- und Metadaten-Information.

class ResNet18Multimodal(nn.Module):
    """
    ResNet18 + Alter als zusätzlicher Input.
    Bild-Features (512) werden mit dem Alter (1) concateniert
    und dann gemeinsam klassifiziert.
    """
    def __init__(self):
        super().__init__()
        basis = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # Backbone: alles außer dem letzten FC Layer
        self.backbone = nn.Sequential(*list(basis.children())[:-1])

        # Finaler Classifier: 512 (Bild) + 1 (Alter) = 513 Eingaben
        self.classifier = nn.Sequential(
            nn.Linear(512 + 1, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, bild, alter):
        # Bild-Features extrahieren
        bild_features = self.backbone(bild)          # (batch, 512, 1, 1)
        bild_features = torch.flatten(bild_features, 1)  # (batch, 512)

        # Bild-Features + Alter zusammenführen
        kombiniert = torch.cat([bild_features, alter], dim=1)  # (batch, 513)

        return self.classifier(kombiniert)


modell_c = ResNet18Multimodal().to(DEVICE)
params_c  = sum(p.numel() for p in modell_c.parameters() if p.requires_grad)
print(f'Modell C – ResNet18 + Alter (Multimodal): {params_c:,} trainierbare Parameter')

In [ ]:
# DataLoader für multimodales Modell (mit_alter=True)
loader_train_c, loader_val_c, _, _ = erstelle_dataloaders(
    df, fuer_resnet18=True, mit_alter=True)

print('=' * 60)
print('TRAINING MODELL C – ResNet18 + Alter (Multimodal)')
print('=' * 60)
history_c = trainiere_modell(
    modell_c, loader_train_c, loader_val_c,
    speicher_pfad=MODELL_PFAD_C, mit_alter=True)

modell_c = lade_bestes_modell(modell_c, MODELL_PFAD_C)
auc_c, fpr_c, tpr_c, cm_c, labels_c, preds_c, meta_c = evaluiere_modell(
    modell_c, loader_val_c, 'Modell C – ResNet18 + Alter', mit_alter=True)

## 11. PCA Visualisierung

In [ ]:
# ── PCA auf ResNet18 Features ─────────────────────────────────────────────────
# ResNet18 lernt 512 Zahlen pro Bild (die Features).
# PCA reduziert diese 512 auf 2 → wir können jeden Punkt in 2D plotten.
# Wenn Krebs und Gesund gut getrennte Cluster bilden → Modell hat gut gelernt.

class FeatureExtractor(nn.Module):
    """ResNet18 ohne FC Layer → gibt 512-dim Features zurück."""
    def __init__(self, basis_modell):
        super().__init__()
        self.backbone = nn.Sequential(*list(basis_modell.children())[:-1])

    def forward(self, x):
        return torch.flatten(self.backbone(x), 1)


extraktor = FeatureExtractor(modell_b).to(DEVICE)
extraktor.eval()

# Teilmenge für PCA (500 Bilder reichen)
df_pca_sample = df.sample(min(500, len(df)), random_state=ZUFALLSZAHL)
pca_loader = DataLoader(
    MammografieDataset(df_pca_sample, BILDER_ORDNER,
                        val_transforms_3kanal, fuer_resnet18=True),
    batch_size=16, shuffle=False
)

alle_features, alle_labels, alle_alter = [], [], []

with torch.no_grad():
    for bilder, labels, meta in pca_loader:
        features = extraktor(bilder.to(DEVICE))
        alle_features.append(features.cpu().numpy())
        alle_labels.extend(labels.numpy())
        alle_alter.extend(
            [v.item() if hasattr(v, 'item') else v for v in meta['age']])

features_matrix = np.vstack(alle_features)
labels_array    = np.array(alle_labels)
alter_array     = np.array(alle_alter, dtype=float)

# PCA auf 2 Komponenten
features_norm = StandardScaler().fit_transform(features_matrix)
pca           = PCA(n_components=2, random_state=ZUFALLSZAHL)
pcs           = pca.fit_transform(features_norm)

print(f'Erklärte Varianz – PC1: {pca.explained_variance_ratio_[0]:.1%} | '
      f'PC2: {pca.explained_variance_ratio_[1]:.1%}')

fig, achsen = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('PCA – ResNet18 Feature-Raum', fontsize=13, fontweight='bold')

# Eingefärbt nach Diagnose
for label, farbe, name in [(0, 'steelblue', 'Gesund'), (1, 'tomato', 'Krebs')]:
    maske = labels_array == label
    achsen[0].scatter(pcs[maske,0], pcs[maske,1],
                      c=farbe, alpha=0.5, s=20, label=name)
achsen[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
achsen[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
achsen[0].set_title('Nach Diagnose')
achsen[0].legend(); achsen[0].grid(alpha=0.3)

# Eingefärbt nach Alter
gueltig = alter_array > 0
sc = achsen[1].scatter(pcs[gueltig,0], pcs[gueltig,1],
                        c=alter_array[gueltig], cmap='plasma', alpha=0.5, s=20)
plt.colorbar(sc, ax=achsen[1], label='Alter')
achsen[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
achsen[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
achsen[1].set_title('Nach Alter')
achsen[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 12. Architektur-Vergleich

In [ ]:
# ── Trainings-Kurven ──────────────────────────────────────────────────────────
fig, achsen = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Architektur-Vergleich', fontsize=13, fontweight='bold')

e = range(1, len(history_a['train_loss']) + 1)
farben   = ['royalblue', 'tomato', 'seagreen']
symbole  = ['-o', '-s', '-^']
modelle  = [
    ('Modell A – CNN', history_a),
    ('Modell B – ResNet18', history_b),
    ('Modell C – ResNet18+Alter', history_c)
]

for titel, metrik, idx in [
    ('Training Loss',    'train_loss', 0),
    ('Validation Loss',  'val_loss',   1),
    ('Validation AUC',   'val_auc',    2)
]:
    for (name, hist), farbe, sym in zip(modelle, farben, symbole):
        achsen[idx].plot(e, hist[metrik], sym, color=farbe, ms=4, label=name)
    achsen[idx].set_title(titel)
    achsen[idx].set_xlabel('Epoche')
    achsen[idx].legend(fontsize=8)
    achsen[idx].grid(alpha=0.3)

plt.tight_layout(); plt.show()


# ── ROC Kurven übereinander ───────────────────────────────────────────────────
plt.figure(figsize=(8, 6))
plt.plot(fpr_a, tpr_a, color='royalblue', lw=2,
         label=f'Modell A – CNN von Null  (AUC = {auc_a:.3f})')
plt.plot(fpr_b, tpr_b, color='tomato',    lw=2,
         label=f'Modell B – ResNet18      (AUC = {auc_b:.3f})')
plt.plot(fpr_c, tpr_c, color='seagreen',  lw=2,
         label=f'Modell C – ResNet18+Alter (AUC = {auc_c:.3f})')
plt.plot([0,1],[0,1],'k--', lw=1, label='Zufall')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Kurven – Vergleich aller Modelle')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


# ── Zusammenfassungstabelle ───────────────────────────────────────────────────
def n_params(modell):
    return sum(p.numel() for p in modell.parameters() if p.requires_grad)

vergleich = pd.DataFrame({
    'Modell':             ['A – CNN von Null', 'B – ResNet18', 'C – ResNet18+Alter'],
    'Trainierb. Param.':  [n_params(modell_a), n_params(modell_b), n_params(modell_c)],
    'Bester Val Loss':    [min(history_a['val_loss']),
                           min(history_b['val_loss']),
                           min(history_c['val_loss'])],
    'Bester Val AUC':     [max(history_a['val_auc']),
                           max(history_b['val_auc']),
                           max(history_c['val_auc'])],
    'Test AUC':           [auc_a, auc_b, auc_c],
})

print('\n=== ZUSAMMENFASSUNG ALLER MODELLE ===')
display(vergleich.style
    .highlight_max(subset=['Bester Val AUC', 'Test AUC'], color='lightgreen')
    .highlight_min(subset=['Bester Val Loss'],             color='lightgreen')
    .format({
        'Bester Val Loss':   '{:.4f}',
        'Bester Val AUC':    '{:.4f}',
        'Test AUC':          '{:.4f}',
        'Trainierb. Param.': '{:,}'
    })
)